# Robo Flow — Auto Label (Google Colab)

## Steps (from Robo Flow app)

1. App me **Open in Colab** dabaya — config URL clipboard me copy ho chuki hai
2. **Cell 2** chalao → paste URL (Ctrl+V) → Run
3. **Runtime → Run all** (ya baqi cells Run karo)
4. Progress **Vercel app** ki Label page pe dekho

Optional: **Runtime → Change runtime type → T4 GPU** (faster)

## Cell 2 — Load config from app (run this first)

In [ ]:
import json
import os
import subprocess
import sys
import urllib.request

# Paste the URL copied when you clicked "Open in Colab" in the app
PREFILL_URL = input("Paste config URL (Ctrl+V) then Enter: ").strip()
if not PREFILL_URL:
    raise SystemExit("No URL — click Open in Colab again in the app")

with urllib.request.urlopen(PREFILL_URL, timeout=60) as resp:
    cfg = json.loads(resp.read().decode("utf-8"))

os.environ["SUPABASE_URL"] = cfg["supabase_url"]
os.environ["SUPABASE_SERVICE_ROLE_KEY"] = cfg["supabase_service_role_key"]
os.environ["HF_TOKEN"] = cfg["hf_token"]
os.environ["HF_DATASET_REPO"] = cfg["hf_dataset_repo"]
os.environ["HF_MODEL_REPO"] = cfg["hf_model_repo"]
os.environ["HF_DATASET_REPO_TYPE"] = "dataset"
os.environ["HF_MODEL_REPO_TYPE"] = "model"
os.environ["RAILWAY_WORKER_URL"] = cfg["railway_url"]
os.environ["WORKER_API_KEY"] = cfg.get("worker_api_key") or ""

PROJECT_ID = cfg["project_id"]
DATASET_ID = cfg["dataset_id"]
MODEL_IDS = cfg["model_ids"]
JOB_ID = cfg.get("job_id") or ""
CONFIDENCE = cfg.get("confidence", 0.15)
IOU = cfg.get("iou", 0.45)
REPO_URL = cfg.get("repo_url", "https://github.com/adeeltariq6480/robo_flow.git")
RAILWAY_URL = cfg.get("railway_url", "")
RELABEL = bool(cfg.get("relabel"))

print("Config loaded")
print("Project:", PROJECT_ID)
print("Dataset:", DATASET_ID)
print("Models:", MODEL_IDS)
if JOB_ID:
    print("Job id (app progress):", JOB_ID)

## Cell 3 — Install (2–5 minutes)

In [ ]:
print("=" * 60)
print("Installing packages — please wait 2–5 minutes…")
print("=" * 60)

!git clone {REPO_URL} robo_flow 2>/dev/null || (cd robo_flow && git pull)
%cd robo_flow/worker
!pip install -q -r requirements.txt

print("Install done. GPU:", end=" ")
!nvidia-smi -L 2>/dev/null || print("CPU only — set Runtime → T4 GPU")

## Cell 4 — Start labeling

In [ ]:
print("=" * 60)
print("Starting auto-label — watch progress in your Vercel app")
print("=" * 60)

cmd = [
    sys.executable,
    "scripts/colab_auto_label.py",
    "--project-id", PROJECT_ID,
    "--dataset-id", DATASET_ID,
    "--model-ids", MODEL_IDS,
    "--confidence", str(CONFIDENCE),
    "--iou", str(IOU),
    "--railway-url", RAILWAY_URL,
]
if JOB_ID:
    cmd.extend(["--job-id", JOB_ID])
if RELABEL:
    cmd.append("--relabel")
if os.environ.get("WORKER_API_KEY"):
    cmd.extend(["--worker-api-key", os.environ["WORKER_API_KEY"]])

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise SystemExit(result.returncode)